# Expected Shortfall (CVaR) - Solution

O **Expected Shortfall** (ES), tambem chamado de **Conditional Value-at-Risk (CVaR)** ou
**Tail Value-at-Risk (TVaR)**, e uma medida de risco que responde:

> *"Dado que a perda excede o VaR, qual e a perda media esperada?"*

Formalmente:

$$\text{ES}_\alpha = E[r_t \mid r_t < \text{VaR}_\alpha]$$

O ES e a medida de risco adotada pelo **Basel III** (2013) como substituto do VaR para
calculo de capital regulatorio em bancos. As razoes incluem:

1. **Subaditividade**: $\text{ES}(A + B) \leq \text{ES}(A) + \text{ES}(B)$ — diversificacao sempre reduz risco
2. O VaR **nao** e subaditivo: portfolios diversificados podem ter VaR *maior* que a soma das partes
3. O ES captura o **tamanho** das perdas extremas, nao apenas a frequencia

**Conteudo:**
1. Definicao de ES
2. ES com t-Student
3. ES com GARCH
4. ES Historico
5. VaR vs ES: subaditividade e Basel III

**Referencias:**
- Artzner, P. et al. (1999). Coherent Measures of Risk. *Mathematical Finance*.
- Acerbi, C. & Tasche, D. (2002). On the coherence of Expected Shortfall. *Journal of Banking & Finance*.
- Basel Committee on Banking Supervision (2013). *Fundamental Review of the Trading Book*.

In [ ]:
import sys

sys.path.insert(0, '..')

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy import stats

from archbox.models import GARCH

%matplotlib inline
plt.rcParams['figure.dpi'] = 100
plt.rcParams['figure.figsize'] = (12, 5)

# Carregar dados
data = pd.read_csv('../data/sp500_returns.csv', parse_dates=['date'], index_col='date')
returns = data['returns']
n = len(returns)
mu = returns.mean()
sigma = returns.std()

print(f"S&P 500: {n} observacoes")
print(f"Media: {mu:.6f}, Desvio padrao: {sigma:.6f}")

## 1. Definicao de ES

Para distribuicao normal, o ES tem formula fechada:

$$\text{ES}_\alpha^{Normal} = \mu - \sigma \cdot \frac{\phi(z_\alpha)}{\alpha}$$

onde $\phi(\cdot)$ e a funcao de densidade da normal padrao e $z_\alpha = \Phi^{-1}(\alpha)$.

Intuicao: o ES e sempre **mais negativo** que o VaR, pois e a media da cauda
(que por definicao esta alem do VaR):

$$|\text{ES}_\alpha| > |\text{VaR}_\alpha|$$

Por exemplo, se VaR(95%) = -1.65%, o ES(95%) pode ser -2.06%.

In [ ]:
# Formula fechada para ES Normal: ES = mu - sigma * phi(z_alpha) / alpha
z_95 = stats.norm.ppf(0.05)
z_99 = stats.norm.ppf(0.01)

# ES Normal (estatico)
es_normal_95 = mu - sigma * stats.norm.pdf(z_95) / 0.05
es_normal_99 = mu - sigma * stats.norm.pdf(z_99) / 0.01

# VaR para comparacao
var_normal_95 = mu + z_95 * sigma
var_normal_99 = mu + z_99 * sigma

print("=== Expected Shortfall - Normal (volatilidade constante) ===\n")
print(f"{'Metrica':<25} {'alpha=5%':>12} {'alpha=1%':>12}")
print("-" * 50)
print(f"{'VaR':<25} {var_normal_95:>12.6f} {var_normal_99:>12.6f}")
print(f"{'ES':<25} {es_normal_95:>12.6f} {es_normal_99:>12.6f}")
print(f"{'Ratio ES/VaR':<25} {es_normal_95/var_normal_95:>12.4f} {es_normal_99/var_normal_99:>12.4f}")

print(f"\nO ES e {abs(es_normal_95/var_normal_95):.1%} do VaR para 95%")
print(f"O ES e {abs(es_normal_99/var_normal_99):.1%} do VaR para 99%")
print("\nPara distribuicao normal, ES(95%)/VaR(95%) = phi(z)/alpha / z = constante")

# Verificar empiricamente
tail_95 = returns[returns < var_normal_95]
tail_99 = returns[returns < var_normal_99]
print("\nVerificacao empirica:")
print(f"  Media dos retornos < VaR(95%): {tail_95.mean():.6f} (ES teorico: {es_normal_95:.6f})")
print(f"  Media dos retornos < VaR(99%): {tail_99.mean():.6f} (ES teorico: {es_normal_99:.6f})")

## 2. ES com t-Student

Para a distribuicao t-Student padronizada com $\nu$ graus de liberdade, o ES tem formula fechada:

$$\text{ES}_\alpha^{t} = \mu + \sigma_t \cdot \left( -\frac{f_\nu(t_\alpha)}{\alpha} \cdot \frac{\nu + t_\alpha^2}{\nu - 1} \cdot \frac{\nu - 2}{\nu} \right)$$

onde $f_\nu$ e a densidade da t-Student e $t_\alpha$ e seu quantil.

Para $\nu$ pequeno, o ES da t-Student e **significativamente maior** em valor absoluto que o da normal,
pois as caudas pesadas implicam perdas extremas mais severas.

In [ ]:
def es_t_student(alpha, nu, mu_val=0, sigma_val=1):
    """ES para t-Student padronizada (variancia unitaria)."""
    t_alpha = stats.t.ppf(alpha, df=nu)
    f_t = stats.t.pdf(t_alpha, df=nu)
    # ES para t-Student padronizada (variancia = (nu-2)/nu)
    # Corrigir para variancia unitaria
    scale = np.sqrt((nu - 2) / nu)
    es_standardized = -(f_t / alpha) * (nu + t_alpha**2) / (nu - 1)
    es = mu_val + sigma_val * es_standardized * scale
    return es

# Testar com diferentes graus de liberdade
print("=== ES com t-Student (estatico) ===\n")
print(f"{'nu':>6} {'VaR(5%)':>12} {'ES(5%)':>12} {'ES/VaR':>10} {'VaR(1%)':>12} {'ES(1%)':>12} {'ES/VaR':>10}")
print("-" * 75)

for nu_val in [4, 5, 6, 8, 10, 15, 30, np.inf]:
    if nu_val == np.inf:
        # Normal
        var5 = mu + z_95 * sigma
        var1 = mu + z_99 * sigma
        es5 = es_normal_95
        es1 = es_normal_99
        label = "Normal"
    else:
        t5 = stats.t.ppf(0.05, df=nu_val) * np.sqrt((nu_val - 2) / nu_val)
        t1 = stats.t.ppf(0.01, df=nu_val) * np.sqrt((nu_val - 2) / nu_val)
        var5 = mu + t5 * sigma
        var1 = mu + t1 * sigma
        es5 = es_t_student(0.05, nu_val, mu, sigma)
        es1 = es_t_student(0.01, nu_val, mu, sigma)
        label = f"{nu_val:.0f}"

    print(f"{label:>6} {var5:>12.6f} {es5:>12.6f} {es5/var5:>10.3f} "
          f"{var1:>12.6f} {es1:>12.6f} {es1/var1:>10.3f}")

print("\nQuanto menor nu, mais pesadas as caudas e maior a diferenca entre ES e VaR.")
print(f"Para nu=4, o ES e ~{abs(es_t_student(0.01, 4, mu, sigma)/( mu + stats.t.ppf(0.01, df=4) * np.sqrt(2/4) * sigma)):.0%} do VaR ao nivel 99%.")

## 3. ES com GARCH

Combinando o GARCH com ES, obtemos uma medida de risco dinamica:

$$\text{ES}_{\alpha,t} = \mu_t + \sigma_t \cdot E[z \mid z < z_\alpha]$$

onde $E[z \mid z < z_\alpha]$ depende da distribuicao assumida para as inovacoes:

- **Normal**: $E[z \mid z < z_\alpha] = -\frac{\phi(z_\alpha)}{\alpha}$
- **t-Student**: $E[z \mid z < z_\alpha] = -\frac{f_\nu(t_\alpha)}{\alpha} \cdot \frac{\nu + t_\alpha^2}{\nu - 1} \cdot \sqrt{\frac{\nu-2}{\nu}}$

O ES dinamico com GARCH-t e o padrao de mercado para calculo de capital regulatorio.

In [ ]:
# Estimar GARCH(1,1) com t-Student
model_t = GARCH(returns.values, p=1, q=1, mean='constant', dist='studentt')
results_t = model_t.fit(disp=False)

sigma_t = results_t.conditional_volatility
mu_t = results_t.params[0]
nu = results_t.params[-1]

print(f"GARCH(1,1)-t estimado: nu = {nu:.2f}")
print(f"Persistencia: {results_t.persistence():.4f}")

# ES dinamico com GARCH-Normal
es_factor_normal_95 = -stats.norm.pdf(z_95) / 0.05
es_factor_normal_99 = -stats.norm.pdf(z_99) / 0.01

# ES dinamico com GARCH-t
t_alpha_95 = stats.t.ppf(0.05, df=nu)
t_alpha_99 = stats.t.ppf(0.01, df=nu)
scale_t = np.sqrt((nu - 2) / nu)

es_factor_t_95 = -(stats.t.pdf(t_alpha_95, df=nu) / 0.05) * (nu + t_alpha_95**2) / (nu - 1) * scale_t
es_factor_t_99 = -(stats.t.pdf(t_alpha_99, df=nu) / 0.01) * (nu + t_alpha_99**2) / (nu - 1) * scale_t

# ES dinamico
es_garch_normal_95 = mu_t + sigma_t * es_factor_normal_95
es_garch_normal_99 = mu_t + sigma_t * es_factor_normal_99
es_garch_t_95 = mu_t + sigma_t * es_factor_t_95
es_garch_t_99 = mu_t + sigma_t * es_factor_t_99

# VaR para comparacao
var_garch_t_95 = mu_t + stats.t.ppf(0.05, df=nu) * scale_t * sigma_t
var_garch_t_99 = mu_t + stats.t.ppf(0.01, df=nu) * scale_t * sigma_t

print("\nES factors (multiplicador de sigma_t):")
print(f"  Normal 95%: {es_factor_normal_95:.4f}")
print(f"  Normal 99%: {es_factor_normal_99:.4f}")
print(f"  t({nu:.0f}) 95%: {es_factor_t_95:.4f}")
print(f"  t({nu:.0f}) 99%: {es_factor_t_99:.4f}")

# Grafico: VaR e ES com GARCH-t
fig, ax = plt.subplots(figsize=(14, 6))
ax.plot(returns.index, returns.values, color='gray', alpha=0.3, linewidth=0.5, label='Retornos')
ax.plot(returns.index, var_garch_t_99, color='orange', linewidth=1, label='VaR 99% (GARCH-t)')
ax.plot(returns.index, es_garch_t_99, color='red', linewidth=1, label='ES 99% (GARCH-t)')
ax.fill_between(returns.index, var_garch_t_99, es_garch_t_99, alpha=0.2, color='red',
                label='Regiao ES - VaR')

mask = returns.values < var_garch_t_99
ax.scatter(returns.index[mask], returns.values[mask],
           color='darkred', s=15, zorder=5, label='Violacoes VaR')

ax.set_title('VaR e Expected Shortfall Dinamicos (GARCH-t, 99%)')
ax.set_ylabel('Retorno')
ax.legend(loc='lower left', fontsize=9)
plt.tight_layout()
plt.show()

## 4. ES Historico

O ES historico e simplesmente a **media dos retornos** que caem abaixo do VaR historico
em uma janela rolling:

$$\text{ES}_{\alpha,t}^{HS} = \frac{1}{|\{i : r_i < \text{VaR}_{\alpha,t}\}|} \sum_{r_i < \text{VaR}_{\alpha,t}} r_i$$

onde a soma e sobre os retornos na janela $[t-W+1, t]$ que sao menores que o VaR.

Este metodo herda as mesmas vantagens e desvantagens do VaR historico
(ghost effects, adaptacao lenta), mas fornece informacao sobre o **tamanho**
das perdas extremas, nao apenas sua frequencia.

In [ ]:
window = 250

es_hist_95 = np.full(n, np.nan)
es_hist_99 = np.full(n, np.nan)
var_hist_95 = np.full(n, np.nan)
var_hist_99 = np.full(n, np.nan)

for t in range(window, n):
    window_returns = returns.values[t - window:t]

    # VaR historico
    var_95 = np.percentile(window_returns, 5)
    var_99 = np.percentile(window_returns, 1)
    var_hist_95[t] = var_95
    var_hist_99[t] = var_99

    # ES historico: media dos retornos abaixo do VaR
    tail_95 = window_returns[window_returns <= var_95]
    tail_99 = window_returns[window_returns <= var_99]

    es_hist_95[t] = tail_95.mean() if len(tail_95) > 0 else var_95
    es_hist_99[t] = tail_99.mean() if len(tail_99) > 0 else var_99

# Estatisticas
valid = ~np.isnan(es_hist_95)
print(f"=== ES Historico (janela = {window} dias) ===\n")
print(f"{'Metrica':<25} {'alpha=5%':>12} {'alpha=1%':>12}")
print("-" * 50)
print(f"{'VaR medio':<25} {var_hist_95[valid].mean():>12.6f} {var_hist_99[valid].mean():>12.6f}")
print(f"{'ES medio':<25} {es_hist_95[valid].mean():>12.6f} {es_hist_99[valid].mean():>12.6f}")
print(f"{'Ratio ES/VaR medio':<25} {(es_hist_95[valid]/var_hist_95[valid]).mean():>12.4f} "
      f"{(es_hist_99[valid]/var_hist_99[valid]).mean():>12.4f}")

# Grafico
fig, ax = plt.subplots(figsize=(14, 6))
ax.plot(returns.index, returns.values, color='gray', alpha=0.3, linewidth=0.5, label='Retornos')
ax.plot(returns.index, var_hist_99, color='orange', linewidth=1, label='VaR Hist. 99%')
ax.plot(returns.index, es_hist_99, color='red', linewidth=1, label='ES Hist. 99%')
ax.fill_between(returns.index, var_hist_99, es_hist_99,
                where=~np.isnan(var_hist_99), alpha=0.2, color='red')

ax.set_title(f'VaR e ES Historicos (janela = {window} dias, 99%)')
ax.set_ylabel('Retorno')
ax.legend(loc='lower left', fontsize=9)
plt.tight_layout()
plt.show()

## 5. VaR vs ES — Por que reguladores preferem ES (Basel III)

O **Comite de Basileia** (BCBS) substituiu o VaR pelo ES no calculo de capital regulatorio
a partir do **Fundamental Review of the Trading Book (FRTB)** de 2013/2016.

### Subaditividade

Uma medida de risco $\rho$ e **subaditiva** se:

$$\rho(A + B) \leq \rho(A) + \rho(B)$$

Isso significa que diversificacao **sempre reduz** (ou nao aumenta) o risco.

- **ES e subaditivo** (medida coerente no sentido de Artzner et al., 1999)
- **VaR nao e subaditivo** — existem casos onde $\text{VaR}(A+B) > \text{VaR}(A) + \text{VaR}(B)$

### Outras vantagens do ES
- Captura o **tamanho** das perdas extremas (VaR so diz se excedeu ou nao)
- Mais **conservador** — requer mais capital, dando margem de seguranca
- Penaliza distribuicoes com **caudas pesadas** (mais realistico para mercados financeiros)

In [ ]:
# Comparacao completa: todos os metodos de ES ao nivel 99%
fig, axes = plt.subplots(2, 1, figsize=(14, 10), sharex=True)

# Painel 1: VaR e ES com GARCH-t
ax = axes[0]
ax.plot(returns.index, returns.values, color='gray', alpha=0.3, linewidth=0.5, label='Retornos')
ax.plot(returns.index, var_garch_t_99, color='orange', linewidth=1.2, label='VaR 99% (GARCH-t)')
ax.plot(returns.index, es_garch_t_99, color='red', linewidth=1.2, label='ES 99% (GARCH-t)')
ax.plot(returns.index, es_garch_normal_99, color='blue', linewidth=1, linestyle='--', label='ES 99% (GARCH-Normal)')
ax.fill_between(returns.index, var_garch_t_99, es_garch_t_99, alpha=0.15, color='red')

mask = returns.values < var_garch_t_99
ax.scatter(returns.index[mask], returns.values[mask],
           color='darkred', s=12, zorder=5, alpha=0.7, label='Violacoes VaR')
ax.set_title('VaR vs ES Dinamicos (GARCH, 99%)')
ax.set_ylabel('Retorno')
ax.legend(loc='lower left', fontsize=8)

# Painel 2: ES historico vs ES GARCH
ax = axes[1]
ax.plot(returns.index, returns.values, color='gray', alpha=0.3, linewidth=0.5, label='Retornos')
ax.plot(returns.index, es_hist_99, color='blue', linewidth=1.2, label='ES Historico 99%')
ax.plot(returns.index, es_garch_t_99, color='red', linewidth=1.2, label='ES GARCH-t 99%')
ax.set_title('ES Historico vs ES GARCH-t (99%)')
ax.set_ylabel('Retorno')
ax.set_xlabel('Data')
ax.legend(loc='lower left', fontsize=9)

plt.tight_layout()
plt.show()

# Tabela resumo comparativa
print("\n" + "=" * 70)
print("Resumo: ES medio por metodo (nivel 99%)")
print("=" * 70)
print(f"{'Metodo':<30} {'ES medio':>12} {'VaR medio':>12} {'ES/VaR':>10}")
print("-" * 65)
print(f"{'Normal (estatico)':<30} {es_normal_99:>12.6f} {var_normal_99:>12.6f} {es_normal_99/var_normal_99:>10.3f}")
print(f"{'GARCH-Normal':<30} {es_garch_normal_99.mean():>12.6f} {(mu_t + z_99*sigma_t).mean():>12.6f} "
      f"{(es_garch_normal_99/(mu_t + z_99*sigma_t)).mean():>10.3f}")
print(f"{'GARCH-t (nu={nu:.0f})':<30} {es_garch_t_99.mean():>12.6f} {var_garch_t_99.mean():>12.6f} "
      f"{(es_garch_t_99/var_garch_t_99).mean():>10.3f}")
print(f"{'Historico (W=250)':<30} {es_hist_99[valid].mean():>12.6f} {var_hist_99[valid].mean():>12.6f} "
      f"{(es_hist_99[valid]/var_hist_99[valid]).mean():>10.3f}")
print("=" * 70)
print("\nBasel III usa ES(97.5%) em vez de VaR(99%) \u2014 o nivel efetivo e equivalente")
print("mas o ES fornece informacao mais completa sobre o risco de cauda.")